# EXP 10 — Low-Resource Neural Machine Translation (English → Assamese)

Fine-tunes `facebook/nllb-200-distilled-600M` (CC-BY-NC-4.0, academic/non-commercial use) on the open **AI4Bharat Samanantar** English–Assamese corpus (CC BY-NC 4.0, downloaded automatically from Hugging Face) and evaluates BLEU/chrF before vs after fine-tuning.

**Kaggle setup:** Settings → Accelerator → GPU (T4 x2 or P100); Internet → **On**. No manual data upload is needed.

## 1. Setup and GPU check

In [ ]:
# Install dependencies
!pip -q install -U transformers datasets evaluate sentencepiece sacrebleu accelerate

import os
import subprocess
import torch

# Fail early if Kaggle GPU is not enabled.
assert torch.cuda.is_available(), (
    'GPU not available. Enable a Kaggle GPU accelerator and restart the session.'
)
print('GPU available:', torch.cuda.get_device_name(0))

# Clone once; rerunning this setup cell reuses the existing clone.
repo = '/kaggle/working/repo'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run([
        'git', 'clone',
        'https://github.com/Kartiken07/LOW-RESOURCE-NEURAL-MACHINE-TRANSLATION.git',
        repo,
    ], check=True)

%cd /kaggle/working/repo


## 2. Load the open dataset and inspect it
Samanantar config `as`: `src` = English, `tgt` = Assamese. (Alternative: `--hf-dataset opus100`.)

In [ ]:
from datasets import load_dataset
raw = load_dataset('ai4bharat/samanantar', 'as', split='train')
print(raw)
for r in raw.select(range(5)): print(r['src'], '=>', r['tgt'])

## 3. Preprocess, deduplicate and split 80:10:10 (seed 42)
Removes empty/duplicate/misaligned pairs, normalises Unicode and whitespace. `MAX_ROWS` keeps a random subset so training fits the Kaggle time limit; set it to `None` to use everything (~140k pairs).

In [ ]:
MAX_ROWS = 40000
!python src/preprocess.py --hf-dataset samanantar --max-rows {MAX_ROWS} --out-dir data/processed

## 4. Tokenization check
Verifies language codes (`eng_Latn` → `asm_Beng`) and that source/target stay aligned.

In [ ]:
from transformers import AutoTokenizer
MODEL = 'facebook/nllb-200-distilled-600M'
tok = AutoTokenizer.from_pretrained(MODEL, src_lang='eng_Latn', tgt_lang='asm_Beng')
train = pd.read_csv('data/processed/train.csv')
for _, r in train.head(3).iterrows():
    enc = tok(r.src, text_target=r.tgt, max_length=128, truncation=True)
    print('SRC:', r.src); print('TGT:', r.tgt)
    print('src tokens:', tok.convert_ids_to_tokens(enc['input_ids'])[:12]); print('tgt tokens:', tok.convert_ids_to_tokens(enc['labels'])[:12]); print()

## 5. Smoke test on a small subset
Run this first to catch errors/OOM cheaply; then run the full training below.

In [ ]:
!python src/train.py --data-dir data/processed --output-dir smoke --max-train-rows 200 --max-val-rows 50 --epochs 1 --batch-size 2

## 6. Baseline: pretrained model before fine-tuning
Use the same `--max-samples` for baseline and fine-tuned model so both are scored on identical examples.

In [ ]:
N_EVAL = 1000  # same value for both evaluations
!python src/evaluate.py --model facebook/nllb-200-distilled-600M --label "Pretrained (before fine-tuning)" --max-samples {N_EVAL}

## 7. Fine-tune (record the actual settings and training time in the README)

In [ ]:
!python src/train.py --data-dir data/processed --output-dir nllb-en-as --epochs 2 --batch-size 4 --grad-accum 4 --lr 2e-5 --max-length 128 --seed 42

## 8. Evaluate the fine-tuned model

In [ ]:
!python src/evaluate.py --model nllb-en-as --label "Fine-tuned" --max-samples {N_EVAL}
print(pd.read_csv('results/metrics.csv'))
pd.read_csv('results/sample_translations.csv')

## 9. Demo: translate an English sentence into Assamese

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM
tok = AutoTokenizer.from_pretrained('nllb-en-as', src_lang='eng_Latn', tgt_lang='asm_Beng')
model = AutoModelForSeq2SeqLM.from_pretrained('nllb-en-as').to('cuda' if torch.cuda.is_available() else 'cpu').eval()
def translate(text):
    b = tok(text, return_tensors='pt', truncation=True, max_length=128).to(model.device)
    with torch.no_grad():
        out = model.generate(**b, forced_bos_token_id=tok.convert_tokens_to_ids('asm_Beng'), max_new_tokens=128, num_beams=4)
    return tok.batch_decode(out, skip_special_tokens=True)[0]
while True:
    s = input('English sentence (blank to stop): ').strip()
    if not s: break
    print('Assamese:', translate(s))

## 10. Download results
Copy `results/metrics.csv` and `results/sample_translations.csv` back into the GitHub repo's `results/` folder, and add the actual scores and training time to the README.